# Inference any Model
Inference any one model using an example of SODA Dataset or custom narrative and characters set.

## Load Libraries

In [ ]:
import sys
from pathlib import Path

project_path = Path.cwd().parent.parent

sys.path.append(str(project_path.resolve()))

In [ ]:
import json
from src.dataset.load_data_soda import SODADataLoader
from src.utils.inferencing import HFModelForInferencing

## Load the SODA Dataset

In [ ]:
soda_dataset_obj = SODADataLoader(
    data_types=['test'],
    samples_per_split=500,
    min_story_length=20,
    max_story_length=250,
    join_dialogue_and_speakers=True,
    keep_speakers_col=True
)
soda_ds = soda_dataset_obj.dataset

d_type = list(soda_ds.keys())[0]

## Load Models

### Get all model details from the config file

In [ ]:
with open('../../config/model_details.json', 'r') as file:
    model_data = json.load(file)

### Create `HFModelForInferencing` object for all the models

In [ ]:
gen_models_obj = {}

for model_type, train_data in model_data.items():
    gen_models_obj[model_type] = {}
    for train_type, train_details in train_data.items():
        if train_details['is-lora'] == True:
            gen_models_obj[model_type][train_type] = HFModelForInferencing(
                hf_model_repo_name=train_details['hf-org-model-path'],
                is_lora=True,
                peft_model_repo_name=train_details['hf-ft-model-path'],
                hf_commit_hash=train_details['hf-commit-id']
            )
        else:
            gen_models_obj[model_type][train_type] = HFModelForInferencing(
                hf_model_repo_name=train_details['hf-ft-model-path'],
                hf_commit_hash=train_details['hf-commit-id']
            )

## Inference the Model

### Select the Example
Just update the value of `i` and it will select the correct example from the SODA Dataset test set.

In [ ]:
i = 3

narrative = soda_ds[d_type][i]['narrative'].split("\n")[0]
actual_dialogue = soda_ds[d_type][i]['dialogue']
characters = soda_ds[d_type][i]['narrative'].split(
    '\n')[1].split(':')[-1].replace('.', '').split(',')
characters = [c.strip() for c in characters]

print("Narrative:", narrative, "-" * 50, sep="\n")
print("Characters:", characters, "-" * 50, sep="\n")
print("Actual Dialogue:", actual_dialogue, "-" * 50, sep="\n")

## Load Model

### Get all model details from the config file

In [ ]:
with open('../../config/model_details.json', 'r') as file:
    all_models_data = json.load(file)

### Show all the available models

In [ ]:
for model_name, model_info in all_models_data.items():
    print(f"Model Name: {model_name}")
    print("   Training Type(s):", end=" ")
    print(*list(model_info.keys()), sep=", ")
    print()

### Select the Model for Inferencing
Just update the `model_name` and `model_training_type` from the list given above.

In [ ]:
model_name = 'BART-LoRA'
model_training_type = 'turn-by-turn'
model_details = all_models_data[model_name][model_training_type]

### Load the selected Model

In [ ]:
if model_details['is_lora'] == True:
    gen_model_obj = HFModelForInferencing(
        hf_model_repo_name=model_details['hf-org-model-path'],
        is_lora=model_details['is_lora'],
        peft_model_repo_name=model_details['hf-ft-model-path'],
        hf_commit_hash=model_details['hf-commit-id']
    )
else:
    gen_model_obj = HFModelForInferencing(
        hf_model_repo_name=model_details['hf-ft-model-path'],
        hf_commit_hash=model_details['hf-commit-id']
    )

## Inference the Model

### Select the inference example type (1 or 2)
1. Inference with custom narrative and characters
2. Inference with an example from the SODA dataset

In [ ]:
inference_example_type = 1

### Give the narrative and characters

In [ ]:
if inference_example_type == 1:
    narrative = "Once upon a time, in a small village nestled between rolling hills, there lived a kind-hearted baker named Clara. She was known far and wide for her delicious bread and pastries, which she baked with love and care every morning. One day, a mysterious traveler arrived in the village, seeking shelter and food. Clara welcomed him into her bakery, offering him a warm loaf of bread and a cup of tea. The traveler was grateful and shared stories of his adventures from distant lands. Inspired by his tales, Clara decided to embark on her own journey, leaving the village behind to explore the world beyond the hills."
    characters = ["Clara", "Mysterious Traveler"]

### Select the Example
Just update the value of `i` and it will select the correct example from the SODA Dataset test set.

In [ ]:
if inference_example_type == 2:
    i = 3

    narrative = soda_ds[d_type][i]['narrative'].split("\n")[0]
    actual_dialogue = soda_ds[d_type][i]['dialogue']
    characters = soda_ds[d_type][i]['narrative'].split(
        '\n')[1].split(':')[-1].replace('.', '').split(',')
    characters = [c.strip() for c in characters]

    print("Narrative:", narrative, "-" * 50, sep="\n")
    print("Characters:", characters, "-" * 50, sep="\n")
    print("Actual Dialogue:", actual_dialogue, "-" * 50, sep="\n")

### Generate the Output
- Update the `prefix_prompt` on for **T5** based models.
- Update the parameters of `generate_dialogue` function to tweak with the generated dialgoues.

In [ ]:
prefix_prompt = "" # Add prefix prompt for T5 model

gen_output = gen_model_obj.generate_dialogue(
    narrative=narrative,
    characters=characters,
    tokenizer_max_length=900,
    max_new_tokens=124,
    prefix_prompt=prefix_prompt,
    gen_turn_by_turn=True,
    max_turns=5
)

print("Generated Dialogue:", gen_output, "-" * 50, sep="\n")